In [1]:
!pip install transformers torch torchvision torchaudio numpy pandas tqdm matplotlib huggingface_hub datasets evaluate scikit-learn accelerate

In [2]:
import sys
import os

# Path to the folder you want to add
subfolder_path = os.path.join(os.getcwd(), "roberta_classifier")

# Add it to sys.path
if subfolder_path not in sys.path:
    sys.path.append(subfolder_path)

In [ ]:
import json
import os
import torch
import sys
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForSequenceClassification,
)

from roberta_classifier.dataset import prepare_datasets
from roberta_classifier.train import train_binary, evaluate_binary
from roberta_classifier.finetuning import save_results, sampling_tuning
from roberta_classifier.seeding import enforce_reproducibility
from roberta_classifier.result_handling import get_balanced_top_configs

results_dir = "roberta_classifier_results"
model_name = "FacebookAI/xlm-roberta-base"
file_base = "_tuning_metrics.csv"

/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
language = "ko"
oversample_ratio, undersample_ratio = get_balanced_top_configs(f"{results_dir}/{language}{file_base}")

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, language)

# ==== MODEL ====
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 1, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets[language])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, language, epoch_history, eval_results)

=== Top Balanced Sampling Configurations ===
╒═════════════════╤══════════════════╤════════════╤═════════════════╤══════════════════╤══════════╤═════════════════╕
│   over_sampling │   under_sampling │   accuracy │   true_accuracy │   false_accuracy │     loss │   balance_score │
╞═════════════════╪══════════════════╪════════════╪═════════════════╪══════════════════╪══════════╪═════════════════╡
│               1 │              0   │   0.980337 │        0.991098 │         0.789474 │ 0.151025 │        0.961543 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│               1 │              1   │   0.980337 │        0.991098 │         0.789474 │ 0.151025 │        0.961543 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│               1 │              0.5 │   0.966292 │        0.976261 │         0.789474 │ 0.21263  │        0.905296 │
╘══════════

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


53.0
[2025-10-25 09:17:07,041] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,F1 Macro,Confusion Matrix
53,No log,0.670484,0.629630,0.970076,0.629630,0.750374,0.433420,"[5, 1, 89, 148]"
106,No log,0.133478,0.958848,0.979868,0.958848,0.966725,0.739270,"[5, 1, 9, 228]"
159,No log,0.129194,0.971193,0.986705,0.971193,0.976284,0.808295,"[6, 0, 7, 230]"
212,No log,0.190502,0.967078,0.985891,0.967078,0.973380,0.791416,"[6, 0, 8, 229]"
265,No log,0.209835,0.971193,0.986705,0.971193,0.976284,0.808295,"[6, 0, 7, 230]"
318,No log,0.218321,0.962963,0.985185,0.962963,0.970541,0.776037,"[6, 0, 9, 228]"
371,No log,0.150806,0.979424,0.988777,0.979424,0.982340,0.847611,"[6, 0, 5, 232]"
424,No log,0.148280,0.975309,0.987654,0.975309,0.979266,0.826923,"[6, 0, 6, 231]"
477,No log,0.152732,0.979424,0.988777,0.979424,0.982340,0.847611,"[6, 0, 5, 232]"
530,0.220800,0.153396,0.979424,0.988777,0.979424,0.982340,0.847611,"[6, 0, 5, 232]"


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84v

[2025-10-25 09:22:36,814] - [INFO] - Training completed.
[2025-10-25 09:22:36,827] - [INFO] - Running evaluation...


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[2025-10-25 09:22:41,308] - [INFO] - Evaluation Results:
[2025-10-25 09:22:41,308] - [INFO] - eval_loss: 0.17167022824287415
[2025-10-25 09:22:41,308] - [INFO] - eval_model_preparation_time: 0.0011
[2025-10-25 09:22:41,309] - [INFO] - eval_accuracy: 0.9691011235955056
[2025-10-25 09:22:41,309] - [INFO] - eval_precision: 0.9939393939393939
[2025-10-25 09:22:41,309] - [INFO] - eval_recall: 0.973293768545994
[2025-10-25 09:22:41,309] - [INFO] - eval_f1: 0.9835082458770614
[2025-10-25 09:22:41,309] - [INFO] - eval_confusion_matrix: [17, 2, 9, 328]
[2025-10-25 09:22:41,309] - [INFO] - eval_runtime: 4.478
[2025-10-25 09:22:41,310] - [INFO] - eval_samples_per_second: 79.5
[2025-10-25 09:22:41,310] - [INFO] - eval_steps_per_second: 10.049
[2025-10-25 09:22:41,315] - [INFO] - Saved per-epoch metrics at: roberta_classifier_results/ko_train_metrics_per_epoch.csv
[2025-10-25 09:22:41,501] - [INFO] - Saved test metrics at: roberta_classifier_results/ko_test_evaluation_metrics.csv


In [ ]:
language = "ar"
oversample_ratio, undersample_ratio = get_balanced_top_configs(f"{results_dir}/{language}{file_base}")

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, language)

# ==== MODEL ====
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 1, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets[language])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, language, epoch_history, eval_results)

=== Top Balanced Sampling Configurations ===
╒═════════════════╤══════════════════╤════════════╤═════════════════╤══════════════════╤═══════════╤═════════════════╕
│   over_sampling │   under_sampling │   accuracy │   true_accuracy │   false_accuracy │      loss │   balance_score │
╞═════════════════╪══════════════════╪════════════╪═════════════════╪══════════════════╪═══════════╪═════════════════╡
│            0.5  │              0.5 │   0.983133 │        0.983471 │         0.980769 │ 0.0947132 │        1        │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼───────────┼─────────────────┤
│            1    │              0.5 │   0.980723 │        0.983471 │         0.961538 │ 0.108005  │        0.987334 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼───────────┼─────────────────┤
│            0.25 │              0   │   0.978313 │        0.980716 │         0.961538 │ 0.0983243 │        0.972187 │
╘══

Map: 100%|██████████| 256/256 [00:00<00:00, 5611.20 examples/s]
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


25.0
[2025-10-25 09:22:47,402] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,F1 Macro,Confusion Matrix
25,No log,0.743257,0.101562,0.010315,0.101562,0.018728,0.092199,"[26, 0, 230, 0]"
50,No log,0.677826,0.734375,0.892228,0.734375,0.784831,0.601027,"[20, 6, 62, 168]"
75,No log,1.175400,0.285156,0.875095,0.285156,0.334438,0.278273,"[24, 2, 181, 49]"
100,No log,0.547881,0.855469,0.921183,0.855469,0.876479,0.728682,"[22, 4, 33, 197]"
125,No log,0.203543,0.941406,0.959401,0.941406,0.946414,0.867837,"[25, 1, 14, 216]"
150,No log,0.236959,0.957031,0.966917,0.957031,0.959772,0.897641,"[25, 1, 10, 220]"
175,No log,0.250783,0.949219,0.962958,0.949219,0.953030,0.882349,"[25, 1, 12, 218]"
200,No log,0.233912,0.960938,0.969069,0.960938,0.963196,0.905605,"[25, 1, 9, 221]"
225,No log,0.239747,0.960938,0.969069,0.960938,0.963196,0.905605,"[25, 1, 9, 221]"
250,No log,0.243245,0.960938,0.969069,0.960938,0.963196,0.905605,"[25, 1, 9, 221]"


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not support

[2025-10-25 09:26:34,281] - [INFO] - Training completed.
[2025-10-25 09:26:34,298] - [INFO] - Running evaluation...


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[2025-10-25 09:26:40,199] - [INFO] - Evaluation Results:
[2025-10-25 09:26:40,199] - [INFO] - eval_loss: 0.11639081686735153
[2025-10-25 09:26:40,199] - [INFO] - eval_model_preparation_time: 0.0013
[2025-10-25 09:26:40,199] - [INFO] - eval_accuracy: 0.980722891566265
[2025-10-25 09:26:40,200] - [INFO] - eval_precision: 0.9944289693593314
[2025-10-25 09:26:40,200] - [INFO] - eval_recall: 0.9834710743801653
[2025-10-25 09:26:40,200] - [INFO] - eval_f1: 0.9889196675900277
[2025-10-25 09:26:40,200] - [INFO] - eval_confusion_matrix: [50, 2, 6, 357]
[2025-10-25 09:26:40,200] - [INFO] - eval_runtime: 5.897
[2025-10-25 09:26:40,200] - [INFO] - eval_samples_per_second: 70.375
[2025-10-25 09:26:40,200] - [INFO] - eval_steps_per_second: 8.818
[2025-10-25 09:26:40,203] - [INFO] - Saved per-epoch metrics at: roberta_classifier_results/ar_train_metrics_per_epoch.csv
[2025-10-25 09:26:40,267] - [INFO] - Saved test metrics at: roberta_classifier_results/ar_test_evaluation_metrics.csv


In [ ]:
language = "te"
oversample_ratio, undersample_ratio = get_balanced_top_configs(f"{results_dir}/{language}{file_base}")

enforce_reproducibility(42)

# ==== PREPARE DATASETS ====
train_set, val_set, test_sets, tokenizer = prepare_datasets(model_name, oversample_ratio, undersample_ratio, language)

# ==== MODEL ====
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

# ==== TRAIN MODEL ====
model, tokenizer, epoch_history = train_binary(model, train_set, val_set, tokenizer, 4, results_dir)

# ==== EVALUATE MODEL ====
eval_results = evaluate_binary(model, tokenizer, test_sets[language])

# ==== SAVE RESULTS & MODEL ====
save_results(results_dir, language, epoch_history, eval_results)

=== Top Balanced Sampling Configurations ===
╒═════════════════╤══════════════════╤════════════╤═════════════════╤══════════════════╤══════════╤═════════════════╕
│   over_sampling │   under_sampling │   accuracy │   true_accuracy │   false_accuracy │     loss │   balance_score │
╞═════════════════╪══════════════════╪════════════╪═════════════════╪══════════════════╪══════════╪═════════════════╡
│             0.5 │              0   │   0.796875 │        0.945017 │         0.333333 │ 0.980083 │        0.695568 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│             0.5 │              1   │   0.796875 │        0.945017 │         0.333333 │ 0.980083 │        0.695568 │
├─────────────────┼──────────────────┼────────────┼─────────────────┼──────────────────┼──────────┼─────────────────┤
│             1   │              0.5 │   0.710938 │        0.810997 │         0.397849 │ 0.927126 │        0.663236 │
╘══════════

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


88.0
[2025-10-25 09:35:31,359] - [INFO] - Starting training of model (answerability classification).


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted,F1 Macro,Confusion Matrix
88,No log,0.300150,0.897059,0.945398,0.897059,0.918314,0.583552,"[2, 3, 11, 120]"
176,No log,0.429520,0.911765,0.956057,0.911765,0.929983,0.643045,"[3, 2, 10, 121]"
264,No log,0.407392,0.963235,0.959726,0.963235,0.961263,0.712717,"[2, 3, 2, 129]"
352,No log,0.385312,0.970588,0.966018,0.970588,0.967023,0.742424,"[2, 3, 1, 130]"
440,No log,0.423701,0.970588,0.971460,0.970588,0.961005,0.659148,"[1, 4, 0, 131]"
528,0.160800,0.385775,0.970588,0.966018,0.970588,0.967023,0.742424,"[2, 3, 1, 130]"
616,0.160800,0.362313,0.977941,0.978435,0.977941,0.973339,0.780054,"[2, 3, 0, 131]"
704,0.160800,0.367055,0.977941,0.978435,0.977941,0.973339,0.780054,"[2, 3, 0, 131]"
792,0.160800,0.366460,0.977941,0.978435,0.977941,0.973339,0.780054,"[2, 3, 0, 131]"
880,0.160800,0.366367,0.977941,0.978435,0.977941,0.973339,0.780054,"[2, 3, 0, 131]"


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/xk84v

[2025-10-25 09:44:11,922] - [INFO] - Training completed.
[2025-10-25 09:44:11,935] - [INFO] - Running evaluation...


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[2025-10-25 09:44:17,086] - [INFO] - Evaluation Results:
[2025-10-25 09:44:17,087] - [INFO] - eval_loss: 1.8895344734191895
[2025-10-25 09:44:17,087] - [INFO] - eval_model_preparation_time: 0.0014
[2025-10-25 09:44:17,087] - [INFO] - eval_accuracy: 0.7786458333333334
[2025-10-25 09:44:17,087] - [INFO] - eval_precision: 0.7959770114942529
[2025-10-25 09:44:17,087] - [INFO] - eval_recall: 0.9518900343642611
[2025-10-25 09:44:17,087] - [INFO] - eval_f1: 0.86697965571205
[2025-10-25 09:44:17,088] - [INFO] - eval_confusion_matrix: [22, 71, 14, 277]
[2025-10-25 09:44:17,088] - [INFO] - eval_runtime: 5.1475
[2025-10-25 09:44:17,088] - [INFO] - eval_samples_per_second: 74.599
[2025-10-25 09:44:17,088] - [INFO] - eval_steps_per_second: 9.325
[2025-10-25 09:44:17,090] - [INFO] - Saved per-epoch metrics at: roberta_classifier_results/te_train_metrics_per_epoch.csv
[2025-10-25 09:44:17,166] - [INFO] - Saved test metrics at: roberta_classifier_results/te_test_evaluation_metrics.csv
